# RealReel : "Real Eyes on Reel Lies"

#### Imports

In [14]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

import joblib
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import cv2  # for video inference


#### Paths

In [15]:
BASE_SPLIT_DIR = r"C:\Users\bhumi\Desktop\RealReel\Datasets\Combined_final_split"  
train_dir = os.path.join(BASE_SPLIT_DIR, "train")
val_dir   = os.path.join(BASE_SPLIT_DIR, "val")
test_dir  = os.path.join(BASE_SPLIT_DIR, "test")

MODEL_DIR = r"C:\Users\bhumi\Desktop\RealReel\Models"
os.makedirs(MODEL_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


#### dataset loaders

In [16]:
img_size = 224

data_transforms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms)
val_dataset   = datasets.ImageFolder(root=val_dir,   transform=data_transforms)
test_dataset  = datasets.ImageFolder(root=test_dir,  transform=data_transforms)

class_names = train_dataset.classes
print("Classes:", class_names)
print("Train images:", len(train_dataset))
print("Val images:",   len(val_dataset))
print("Test images:",  len(test_dataset))

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=2)


Classes: ['fake', 'real']
Train images: 28964
Val images: 6238
Test images: 6196


#### MobileNetV2 feature extraction

In [17]:
try:
    from torchvision.models import MobileNet_V2_Weights
    mobilenet = models.mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
except Exception:
    mobilenet = models.mobilenet_v2(pretrained=True)

mobilenet.classifier = nn.Identity()
mobilenet = mobilenet.to(device)
mobilenet.eval()
print("Feature extractor ready.")


Feature extractor ready.


#### MobileNetV2 feature extraction

In [8]:
try:
    from torchvision.models import MobileNet_V2_Weights
    mobilenet = models.mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
except Exception:
    mobilenet = models.mobilenet_v2(pretrained=True)

mobilenet.classifier = nn.Identity()
mobilenet = mobilenet.to(device)
mobilenet.eval()
print("Feature extractor ready.")


Feature extractor ready.


#### feature extraction

In [18]:
def extract_features(dataloader, model, device, desc="Extracting"):
    all_features = []
    all_labels = []

    for images, labels in tqdm(dataloader, desc=desc):
        images = images.to(device)
        with torch.no_grad():
            feats = model(images)
        feats = feats.cpu().numpy()

        all_features.append(feats)
        all_labels.append(labels.numpy())

    all_features = np.concatenate(all_features, axis=0)
    all_labels   = np.concatenate(all_labels, axis=0)
    return all_features, all_labels

X_train, y_train = extract_features(train_loader, mobilenet, device,
                                    desc="Train feature extraction")
X_val,   y_val   = extract_features(val_loader,   mobilenet, device,
                                    desc="Val feature extraction")
X_test,  y_test  = extract_features(test_loader,  mobilenet, device,
                                    desc="Test feature extraction")

print("Train features:", X_train.shape)
print("Val features:",   X_val.shape)
print("Test features:",  X_test.shape)


Train feature extraction:   0%|          | 0/906 [00:13<?, ?it/s]

Val feature extraction:   0%|          | 0/195 [00:13<?, ?it/s]

Test feature extraction:   0%|          | 0/194 [00:13<?, ?it/s]

Train features: (28964, 1280)
Val features: (6238, 1280)
Test features: (6196, 1280)


#### Training linear SVC + Evaluate

In [19]:
svc = LinearSVC(C=1.0, max_iter=5000)

print("Training Linear SVC...")
with tqdm(total=1, desc="Fitting LinearSVC") as pbar:
    svc.fit(X_train, y_train)
    pbar.update(1)


Training Linear SVC...


Fitting LinearSVC:   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
# Validation
y_val_pred = []
for i in tqdm(range(0, len(X_val), 512), desc="SVC val prediction"):
    batch = X_val[i:i+512]
    preds = svc.predict(batch)
    y_val_pred.extend(preds)
y_val_pred = np.array(y_val_pred)

val_acc = accuracy_score(y_val, y_val_pred)
print("Validation Accuracy: {:.2f}%".format(val_acc * 100))
print("\nValidation classification report:")
print(classification_report(y_val, y_val_pred, target_names=class_names))


SVC val prediction:   0%|          | 0/13 [00:00<?, ?it/s]

Validation Accuracy: 91.01%

Validation classification report:
              precision    recall  f1-score   support

        fake       0.92      0.90      0.91      3132
        real       0.90      0.92      0.91      3106

    accuracy                           0.91      6238
   macro avg       0.91      0.91      0.91      6238
weighted avg       0.91      0.91      0.91      6238



In [21]:
# Test
y_test_pred = []
for i in tqdm(range(0, len(X_test), 512), desc="SVC test prediction"):
    batch = X_test[i:i+512]
    preds = svc.predict(batch)
    y_test_pred.extend(preds)
y_test_pred = np.array(y_test_pred)

test_acc = accuracy_score(y_test, y_test_pred)
print("Test Accuracy: {:.2f}%".format(test_acc * 100))
print("\nTest classification report:")
print(classification_report(y_test, y_test_pred, target_names=class_names))


SVC test prediction:   0%|          | 0/13 [00:00<?, ?it/s]

Test Accuracy: 91.14%

Test classification report:
              precision    recall  f1-score   support

        fake       0.92      0.90      0.91      3091
        real       0.90      0.92      0.91      3105

    accuracy                           0.91      6196
   macro avg       0.91      0.91      0.91      6196
weighted avg       0.91      0.91      0.91      6196



#### Save Models

In [22]:
mobilenet_path = os.path.join(MODEL_DIR, "mobilenet_feature_extractor_split.pth")
svc_path       = os.path.join(MODEL_DIR, "linear_svc_mobilenet_split.joblib")

torch.save(mobilenet.state_dict(), mobilenet_path)
joblib.dump(svc, svc_path)

print("Saved:", mobilenet_path)
print("Saved:", svc_path)


Saved: C:\Users\bhumi\Desktop\RealReel\Models\mobilenet_feature_extractor_split.pth
Saved: C:\Users\bhumi\Desktop\RealReel\Models\linear_svc_mobilenet_split.joblib


#### Prediction on single images

In [23]:
def predict_single_image(img_path, model, svc, transform, device, class_names):
    img = Image.open(img_path).convert("RGB")
    x = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        feat = model(x).cpu().numpy()

    pred_idx = int(svc.predict(feat)[0])
    label = class_names[pred_idx]
    return label, pred_idx

# Example
test_img = r"C:\Users\bhumi\Desktop\RealReel\Datasets\Combined_final_split\test\real\afoovlsmtx_250_0.png"  
label, idx = predict_single_image(test_img, mobilenet, svc,
                                  data_transforms, device, class_names)
print("Prediction:", label)


Prediction: real


In [24]:
def predict_single_image(img_path, model, svc, transform, device, class_names):
    img = Image.open(img_path).convert("RGB")
    x = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        feat = model(x).cpu().numpy()

    pred_idx = int(svc.predict(feat)[0])
    label = class_names[pred_idx]
    return label, pred_idx

# Example
test_img = r"C:\Users\bhumi\Desktop\RealReel\Datasets\Combined_final_split\test\fake\ahdbuwqxit_79_0.png"  
label, idx = predict_single_image(test_img, mobilenet, svc,
                                  data_transforms, device, class_names)
print("Prediction:", label)


Prediction: fake


#### Prediction on Video

In [25]:
def analyze_video(video_path,
                  model,
                  svc,
                  transform,
                  device,
                  class_names,
                  frame_skip=10,
                  max_frames=200):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    preds = []
    idx = 0
    processed = 0

    total_steps = total_frames // frame_skip if total_frames > 0 else None
    pbar = tqdm(total=total_steps, desc="Analyzing video", unit="frame")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if idx % frame_skip != 0:
            idx += 1
            continue

        if max_frames is not None and processed >= max_frames:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # center crop to square
        h, w, _ = frame_rgb.shape
        side = min(h, w)
        y1 = (h - side) // 2
        x1 = (w - side) // 2
        crop = frame_rgb[y1:y1+side, x1:x1+side]

        pil_img = Image.fromarray(crop)
        x = transform(pil_img).unsqueeze(0).to(device)

        with torch.no_grad():
            feat = model(x).cpu().numpy()

        pred_idx = int(svc.predict(feat)[0])
        preds.append(pred_idx)

        processed += 1
        idx += 1

        if total_steps is not None:
            pbar.update(1)

    cap.release()
    pbar.close()

    if len(preds) == 0:
        raise RuntimeError("No frames processed; try lowering frame_skip.")

    preds = np.array(preds)
    values, counts = np.unique(preds, return_counts=True)
    majority_idx = int(values[np.argmax(counts)])
    majority_label = class_names[majority_idx]

    fake_idx = class_names.index("fake")
    fake_ratio = float(np.mean(preds == fake_idx))

    return {
        "video_path": video_path,
        "frames_processed": processed,
        "majority_label": majority_label,
        "fake_ratio": fake_ratio,
        "class_counts": {class_names[int(v)]: int(c)
                         for v, c in zip(values, counts)}
    }


In [26]:
video_path = r"C:\Users\bhumi\Desktop\RealReel\Datasets\Combined_final_split\test\fake\654_648.mp4"
res = analyze_video(video_path, mobilenet, svc,
                    data_transforms, device, class_names,
                    frame_skip=10, max_frames=200)

print("Video:", res["video_path"])
print("Frames processed:", res["frames_processed"])
print("Majority label:", res["majority_label"].upper())
print("Fake ratio:", f"{res['fake_ratio']:.2f}")
print("Class counts:", res["class_counts"])


Analyzing video:   0%|          | 0/41 [00:00<?, ?frame/s]

Video: C:\Users\bhumi\Desktop\RealReel\Datasets\Combined_final_split\test\fake\654_648.mp4
Frames processed: 41
Majority label: FAKE
Fake ratio: 1.00
Class counts: {'fake': 41}


In [29]:
video_path = r"C:\Users\bhumi\Desktop\RealReel\Datasets\Combined_final_split\test\real\24__talking_angry_couch.mp4"
res = analyze_video(video_path, mobilenet, svc,
                    data_transforms, device, class_names,
                    frame_skip=10, max_frames=200)

print("Video:", res["video_path"])
print("Frames processed:", res["frames_processed"])
print("Majority label:", res["majority_label"].upper())
print("Fake ratio:", f"{res['fake_ratio']:.2f}")
print("Class counts:", res["class_counts"])


Analyzing video:   0%|          | 0/124 [00:00<?, ?frame/s]

Video: C:\Users\bhumi\Desktop\RealReel\Datasets\Combined_final_split\test\real\24__talking_angry_couch.mp4
Frames processed: 125
Majority label: REAL
Fake ratio: 0.00
Class counts: {'real': 125}
